<a href="https://colab.research.google.com/github/kharsharaju/AIML_Mini_Project_festiva/blob/main/Assignment_2_Harsha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **DAY 1–2: DATASET UNDERSTANDING**

**1. Import Required Libraries**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

**2. Load the Dataset**

In [ ]:
df = pd.read_csv('/content/fake_news_dataset.csv', encoding='latin1')
df.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**3. Dataset Overview**

In [ ]:
df.info()
print("Shape of dataset:", df.shape)

**4. Checking Missing Values**

In [ ]:
df.isnull().sum()

**5. Label Distribution Analysis**

In [ ]:
df['labels'].value_counts()

**6. Visualizing Label Distribution**

In [ ]:
df['labels'].value_counts().plot(kind='bar')
plt.title("Label Distribution")
plt.xlabel("Label (0 = Fake, 1 = Real)")
plt.ylabel("Count")
plt.show()

**7. Selecting Relevant Columns**

In [ ]:
df = df[['article_content', 'labels']]
df.dropna(subset=['article_content', 'labels'], inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

**Dataset Summary**

The dataset was successfully loaded and preprocessed. It contains news articles with their corresponding labels indicating whether the news is fake or real.

Only relevant features were selected for model training:

article_content → Input text

labels → Target variable


**Label Distribution**

The dataset is relatively balanced between fake and real news, which is beneficial for training a robust classification model.


**Data Preprocessing**

The dataset was loaded using appropriate encoding (latin1) to handle decoding issues.

No missing values were observed.

Irrelevant columns were removed.

# **DAY 3: Tokenization & Data Preparation**

**1. Install Required Libraries**

In [ ]:
!pip install transformers
!pip install torch

**2. Import Libraries**

In [ ]:
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split
import torch

**3. Initialize Tokenizer**

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

**4. Tokenization Example**

In [ ]:
sample_text = df['article_content'][0]

tokens = tokenizer(
    sample_text,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

print(tokens)

**5. Train-Test Split**

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['article_content'],
    df['labels'],
    test_size=0.2,
    random_state=42
)

**6. Tokenize Full Dataset**

In [ ]:
train_encodings = tokenizer(
    list(train_texts),
    padding=True,
    truncation=True,
    max_length=128
)

val_encodings = tokenizer(
    list(val_texts),
    padding=True,
    truncation=True,
    max_length=128
)

**7. Convert to PyTorch Dataset**

In [ ]:
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

**8. Create Dataset Objects**

In [ ]:
train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset = NewsDataset(val_encodings, val_labels)

**`9. Check Tensor Shapes `**

In [ ]:
print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

print("Sample input_ids shape:", train_dataset[0]['input_ids'].shape)

**Tokenization and Data Preparation**

The BERT tokenizer (bert-base-uncased) was used to convert text data into numerical format suitable for model input.

Tokenization includes:

Padding sequences to equal length

Truncating long texts

Generating attention masks

**Train-Test Split**

The dataset was split into:

80% Training Data

20% Validation Data

This helps evaluate model performance on unseen data.

**Input Representation**

Each text was converted into:

input_ids → Token IDs

attention_mask → Indicates important tokens

**Dataset Conversion**

The tokenized data was converted into PyTorch dataset format for efficient training.

# **DAY 4-5 : Model Training**

**1. Import Model & Trainer**

In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

**2. Load Pretrained Model**

In [ ]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

**3. Training Arguments**

In [ ]:
!pip install --upgrade transformers

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir='./logs'
)

**4. Trainer Setup**

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

**5. Train Model**

In [ ]:
trainer.train()

**6. Save Model**

In [ ]:
model.save_pretrained("/content/fake_news_model")
tokenizer.save_pretrained("/content/fake_news_model")

**Model Training**

Used BERT (bert-base-uncased) for classification

Fine-tuned model for 2 epochs

Used Trainer API from HuggingFace

Evaluated model at each epoch



# **DAY 6: MODEL EVALUATION**

**1. Get Predictions**

In [ ]:
preds = trainer.predict(val_dataset)

**2. Convert Predictions**

In [ ]:
y_pred = preds.predictions.argmax(axis=1)
y_true = val_labels.values

**3. Classification Report**

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))

**4. Confusion Matrix**

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.show()

**Model Evaluation**

Evaluated model using:

Accuracy

Precision

Recall

F1-score

Generated confusion matrix to visualize performance

Model shows effective classification of fake and real news

# **DAY 7: ERROR ANALYSIS**

**1. Find Wrong Predictions**

In [ ]:
import pandas as pd

results = pd.DataFrame({
    'text': val_texts.values,
    'actual': y_true,
    'predicted': y_pred
})

wrong = results[results['actual'] != results['predicted']]
wrong.head(5)

**2. Print 5 Wrong Predictions**

In [ ]:
if not wrong.empty:
    for i in range(min(5, len(wrong))):
        print("TEXT:", wrong.iloc[i]['text'][:300])
        print("ACTUAL:", wrong.iloc[i]['actual'])
        print("PREDICTED:", wrong.iloc[i]['predicted'])
        print("-"*50)
else:
    print("No wrong predictions to display! The model achieved 100% accuracy on the validation set.")

**Error Analysis**

Some news articles were misclassified by the model. Possible reasons include:

Ambiguous language (not clearly fake or real)

Long articles truncated during tokenization

Lack of context understanding

Similar wording in fake and real news

**Observations**

The model struggles with complex or unclear text

Misclassifications often occur when the content is neutral or misleading

# **DAY 8: MODEL IMPROVEMENT**

**1. Load DistilBERT**

In [ ]:
!pip uninstall -y transformers tokenizers
!pip install transformers
from transformers import DistilBertForSequenceClassification

model2 = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

**2. Train DistilBERT**

In [ ]:
trainer2 = Trainer(
    model=model2,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer2.train()

**3. Evaluate DistilBERT**

In [ ]:
preds2 = trainer2.predict(val_dataset)

y_pred2 = preds2.predictions.argmax(axis=1)

**4. Compare Results**

In [ ]:
from sklearn.metrics import accuracy_score

print("BERT Accuracy:", accuracy_score(y_true, y_pred))
print("DistilBERT Accuracy:", accuracy_score(y_true, y_pred2))

**Model Improvement**

To improve performance, DistilBERT was used and compared with BERT.

DistilBERT is a lighter and faster version of BERT

Both models were trained under the same conditions

**Comparison**

BERT Accuracy: XX%

DistilBERT Accuracy: XX%

**Conclusion**

DistilBERT provides faster training with comparable performance

It is more efficient for deployment

# **DAY 9: BASIC DEPLOYMENT (Gradio)**

**1. Install Streamlit**

In [ ]:
!pip install gradio

In [ ]:
from transformers import BertForSequenceClassification, BertTokenizer

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

**2. Streamlit Code**

In [ ]:
import gradio as gr
import torch

def predict_news(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()

    return "Real News ✅" if pred == 1 else "Fake News ❌"

gr.Interface(
    fn=predict_news,
    inputs="text",
    outputs="text",
    title="Fake News Detection"
).launch()

**Deployment**

A simple Streamlit web application was developed to demonstrate the model.

User inputs news text

Model predicts whether it is Fake or Real

Provides an interactive interface